# 🌿 PlantGuard — Multimodal Plant Disease Detection

**Offline-First ML Pipeline**: Vision + Audio + Text for plant disease detection and care advice.

This notebook demonstrates:
- **Vision**: ResNet50 fine-tuned on PlantVillage dataset
- **Audio**: Whisper-tiny + CNN-LSTM for speech-to-disease classification
- **Text**: DistilBERT for plant care Q&A
- **UI**: Streamlit with real-time audio/video support

**Privacy**: All processing happens locally — no cloud APIs or external data transmission.

## 🔧 Environment Setup

### For Google Colab Users

In [ ]:
# Only run this cell in Google Colab
import os
import subprocess
import sys

if "google.colab" in sys.modules:
    print("🔄 Setting up PlantGuard in Google Colab...")

    # Clone the private repository using token authentication
    try:
        # Install python-dotenv if not available
        try:
            from dotenv import load_dotenv
        except ImportError:
            subprocess.run(["pip", "install", "python-dotenv"], check=True)
            from dotenv import load_dotenv

        # Load environment variables
        load_dotenv()
        github_token = os.getenv("GITHUB_TOKEN")

        if github_token:
            clone_url = f"https://{github_token}@github.com/arslanmit/PlantGuard.git"
        else:
            clone_url = "https://github.com/arslanmit/PlantGuard.git"
            print("⚠️  No GitHub token found - you may need to authenticate manually")

        subprocess.run(["git", "clone", clone_url], check=True)
        os.chdir("PlantGuard")

        # Install dependencies
        subprocess.run(["pip", "install", "-r", "requirements.txt"], check=True)
        subprocess.run(["pip", "install", "-e", "."], check=True)

        print("✅ PlantGuard setup complete!")
    except subprocess.CalledProcessError as e:
        print(f"❌ Setup failed: {e}")
        print("💡 Try running the shell commands manually in separate cells")
else:
    print("📍 Running locally - ensure you've run 'make setup' first")

**Alternative: Manual Setup (if above fails)**

Run these commands in separate cells if the automated setup doesn't work:

In [ ]:
# Run each line in a separate cell if needed
# !git clone https://github.com/arslanmit/PlantGuard.git
# %cd PlantGuard
# !pip install -r requirements.txt
# !pip install -e .
print("💡 Uncomment and run the commands above in separate cells if needed")

### Import Core Modules

In [ ]:
import tempfile
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image
from torch.utils.tensorboard import SummaryWriter

from src.core.nlp import answer
from src.core.vision import CLASSES as VISION_CLASSES

# PlantGuard core modules
from src.core.vision import load_image_model, predict_image

warnings.filterwarnings("ignore")
print(f"🔥 PyTorch device: {torch.cuda.get_device_name() if torch.cuda.is_available() else 'CPU'}")
print(f"🌿 Vision classes: {VISION_CLASSES}")

## 🖼️ Vision Module Demo

Test the ResNet50-based plant disease detection system.

In [ ]:
def demo_vision_prediction(image_path: str) -> dict[str, float]:
    """Demonstrate vision-based disease prediction."""
    try:
        # Load and display image
        img = Image.open(image_path).convert("RGB")

        plt.figure(figsize=(8, 6))
        plt.imshow(img)
        plt.axis("off")
        plt.title(f"Input Image: {Path(image_path).name}")
        plt.show()

        # Predict disease
        predictions = predict_image(img)

        # Display results
        print("\n🔍 Disease Detection Results:")
        for disease, confidence in sorted(predictions.items(), key=lambda x: x[1], reverse=True):
            print(f"  {disease}: {confidence:.3f} ({confidence * 100:.1f}%)")

        return predictions

    except Exception as e:
        print(f"❌ Vision prediction failed: {e}")
        return {}


# Test with a sample image (create a dummy image if no real data available)
print("📸 Testing vision module...")

# Create a sample leaf image for testing
sample_img = Image.new("RGB", (224, 224), color="green")
with tempfile.NamedTemporaryFile(suffix=".jpg", delete=False) as tmp:
    sample_img.save(tmp.name)
    sample_path = tmp.name

vision_results = demo_vision_prediction(sample_path)

# Clean up
Path(sample_path).unlink(missing_ok=True)

## 🎙️ Audio Module Demo

Test speech recognition and audio-based disease classification.

In [ ]:
def demo_audio_processing(text_input: str) -> tuple[str, str]:
    """Demonstrate audio processing with text simulation."""
    try:
        print(f"🎤 Simulating audio input: '{text_input}'")

        # Simulate transcription (in real use, this would be from audio file)
        from src.core.audio import classify_from_transcript

        transcription = text_input
        classification = classify_from_transcript(transcription)

        print(f"📝 Transcription: {transcription}")
        print(f"🔍 Predicted disease: {classification}")

        return transcription, classification

    except Exception as e:
        print(f"❌ Audio processing failed: {e}")
        return "", "healthy"


# Test different audio scenarios
print("🎵 Testing audio module...")

test_phrases = [
    "I see white powder on my plant leaves",
    "There are brown spots appearing on the foliage",
    "Orange rust-like pustules on leaf undersides",
    "My plant looks healthy and green",
]

for phrase in test_phrases:
    print(f"\n{'=' * 50}")
    demo_audio_processing(phrase)

## 💬 NLP Module Demo

Test the DistilBERT-based plant care Q&A system.

In [ ]:
def demo_plant_qa(question: str) -> str:
    """Demonstrate plant care question answering."""
    try:
        print(f"❓ Question: {question}")

        response = answer(question)

        print(f"💡 Answer: {response}")
        return response

    except Exception as e:
        print(f"❌ Q&A failed: {e}")
        return "Unable to process question."


# Test plant care questions
print("🌱 Testing NLP Q&A module...")

test_questions = [
    "How do I treat powdery mildew?",
    "What causes brown spots on leaves?",
    "How can I prevent rust disease?",
    "What is the best watering schedule?",
]

for question in test_questions:
    print(f"\n{'=' * 60}")
    demo_plant_qa(question)

## 🔬 Model Training Demo

Demonstrate how to set up training with TensorBoard logging.

In [ ]:
def setup_training_demo() -> None:
    """Demonstrate training setup with TensorBoard integration."""
    from datetime import datetime

    # Create unique run directory
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    run_dir = f"./runs/plantguard_demo_{timestamp}"

    print(f"📊 Setting up TensorBoard logging in: {run_dir}")

    # Initialize TensorBoard writer
    writer = SummaryWriter(run_dir)

    # Simulate training metrics
    print("🏋️ Simulating training metrics...")

    for epoch in range(5):
        # Simulate decreasing loss
        train_loss = 2.0 * np.exp(-epoch * 0.3) + np.random.normal(0, 0.1)
        val_loss = 2.2 * np.exp(-epoch * 0.25) + np.random.normal(0, 0.15)

        # Simulate increasing accuracy
        train_acc = 0.3 + 0.6 * (1 - np.exp(-epoch * 0.4)) + np.random.normal(0, 0.02)
        val_acc = 0.25 + 0.65 * (1 - np.exp(-epoch * 0.35)) + np.random.normal(0, 0.03)

        # Log metrics
        writer.add_scalar("Loss/Train", train_loss, epoch)
        writer.add_scalar("Loss/Validation", val_loss, epoch)
        writer.add_scalar("Accuracy/Train", train_acc, epoch)
        writer.add_scalar("Accuracy/Validation", val_acc, epoch)

        print(f"Epoch {epoch + 1}: Train Loss={train_loss:.3f}, Val Acc={val_acc:.3f}")

    writer.close()

    print("\n✅ Training demo complete!")
    print(f"📈 View results: tensorboard --logdir {run_dir}")

    return run_dir


# Run training demo
training_dir = setup_training_demo()

## 🚀 Streamlit App Launch

Launch the full multimodal Streamlit application.

In [ ]:
def launch_streamlit_app() -> None:
    """Launch the PlantGuard Streamlit application."""
    import subprocess
    import sys
    import time

    print("🚀 Launching PlantGuard Streamlit App...")

    # Check if running in Colab
    if "google.colab" in sys.modules:
        print("📍 Google Colab detected - setting up tunnel...")

        # Kill any existing processes on port 8501
        try:
            subprocess.run(["fuser", "-k", "8501/tcp"], check=False, capture_output=True)
        except:
            pass

        # Start Streamlit in background
        cmd = [
            "streamlit",
            "run",
            "src/ui/app_streamlit.py",
            "--server.address",
            "0.0.0.0",
            "--server.port",
            "8501",
            "--server.headless",
            "true",
        ]

        print("⚡ Starting Streamlit server...")
        process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE)

        # Wait for server to start
        time.sleep(5)

        # Setup Cloudflare tunnel
        try:
            from pycloudflared import try_cloudflare

            public_url = try_cloudflare(8501)
            print("\n🌐 PlantGuard is now accessible at:")
            print(f"   {public_url}")
            print("\n🎯 Features available:")
            print("   📸 Image Upload: Plant disease detection")
            print("   🎙️ Voice Input: Real-time microphone + file upload")
            print("   💬 Text Q&A: Plant care questions")

        except ImportError:
            print("❌ pycloudflared not available. Install with: pip install pycloudflared")
            print("🔗 Local access: http://localhost:8501")

    else:
        print("💻 Local environment detected")
        print("🔧 Run the following command in your terminal:")
        print("   make run")
        print("   # or directly: streamlit run src/ui/app_streamlit.py")
        print("\n🔗 Then visit: http://localhost:8501")


# Launch the app
launch_streamlit_app()

## 🔒 Privacy & Security Notes

**PlantGuard Privacy Guarantees:**

✅ **Offline Processing**: All ML inference happens locally  
✅ **No Cloud APIs**: Whisper-tiny, ResNet50, DistilBERT run on-device  
✅ **No Data Transmission**: User images/audio never leave your environment  
✅ **Temporary Files**: Audio files cleaned up immediately after processing  
✅ **Session Scope**: No persistent storage of user data  

**Security Features:**
- Pinned model revisions for reproducibility
- Input validation (file size, format, duration limits)
- Graceful error handling with fallbacks
- No external network dependencies for core functionality

## 📊 Performance Monitoring

Monitor system performance and model behavior.

In [ ]:
def system_diagnostics() -> None:
    """Run system diagnostics for PlantGuard."""
    import platform

    import psutil

    print("🔍 PlantGuard System Diagnostics")
    print("=" * 40)

    # System info
    print(f"🖥️  Platform: {platform.system()} {platform.release()}")
    print(f"🐍 Python: {platform.python_version()}")
    print(f"💾 RAM: {psutil.virtual_memory().total / (1024**3):.1f} GB")
    print(f"⚡ CPU: {psutil.cpu_count()} cores")

    # PyTorch info
    print(f"🔥 PyTorch: {torch.__version__}")
    print(f"🎮 CUDA Available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"🎯 GPU: {torch.cuda.get_device_name()}")
        print(
            f"📊 GPU Memory: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.1f} GB"
        )

    # Model loading test
    print("\n🧪 Model Loading Tests:")

    try:
        model = load_image_model()
        param_count = sum(p.numel() for p in model.parameters())
        print(f"✅ Vision Model: {param_count:,} parameters")
    except Exception as e:
        print(f"❌ Vision Model: {e}")

    try:
        from src.core.audio import asr_pipe

        pipe = asr_pipe()
        print(f"✅ Audio Model: {pipe.model.config.name_or_path}")
    except Exception as e:
        print(f"❌ Audio Model: {e}")

    try:
        from src.core.nlp import qa_pipe

        pipe = qa_pipe()
        print(f"✅ NLP Model: {pipe.model.config.name_or_path}")
    except Exception as e:
        print(f"❌ NLP Model: {e}")

    print("\n🎯 All systems ready for plant disease detection!")


# Run diagnostics
system_diagnostics()

## 🎯 Next Steps

**Model Training:**
1. **Vision**: Fine-tune ResNet50 on PlantVillage dataset
2. **Audio**: Train CNN-LSTM on plant disease audio descriptions
3. **Fusion**: Combine modalities with MLP fusion layer

**Data Collection:**
- Gather real plant disease images
- Record audio descriptions in multiple languages
- Expand FAQ context for better Q&A

**Deployment:**
- Package models in `data/` directory
- Test offline functionality thoroughly
- Optimize for mobile/edge deployment

**Quality Assurance:**
```bash
make lint    # Code quality checks
make test    # Run test suite
make run     # Launch application
```

Happy plant disease detection! 🌿🔬